# Express.js

Express is a minimalist, flexible, and unopinionated web application framework for Node.js, designed to simplify building server-side applications, websites, and RESTful APIs. Instead of interacting with the raw Node `http` module — which requires a lot of boilerplate — Express provides a clean abstraction layer over network communication.

## 🚀 Key Technical Features

- **Robust routing** — Matches HTTP verbs (`GET`, `POST`, `PUT`, `PATCH`, `DELETE`) to URL paths.
- **Middleware architecture** — Runs modular functions in sequence across the request/response cycle.
- **Dynamic content views** — Plugs into templating engines (EJS, Pug, Handlebars) to render HTML on the server.
- **Database agnostic** — Works with MongoDB, PostgreSQL, MySQL, or anything else; Express itself has no opinion.

## 🛠️ Step-by-Step Implementation Guide

### 1. Project Initialization

```bash
mkdir express-server
cd express-server
npm init -y
npm install express
```

### 2. Writing the Server Code

Create `app.js` in the project root:

```javascript
// Import the Express module
const express = require('express');

// Create an app instance
const app = express();
const PORT = 3000;

// Parse incoming JSON request bodies
app.use(express.json());

// A GET handler for the root path
app.get('/', (req, res) => {
    res.send('Welcome to your Express backend server.');
});

// Start listening
app.listen(PORT, () => {
    console.log(`Server running at http://localhost:${PORT}`);
});
```

### 3. Running It

```bash
node app.js
```

In practice, use `node --watch app.js` (built into modern Node) or `nodemon` so the server restarts automatically when you save a file. Add it as a script:

```json
{
  "scripts": {
    "dev": "node --watch app.js",
    "start": "node app.js"
  }
}
```

## 🗺️ Primary Foundational Concepts

### Request and Response Lifecycle

Every route handler receives two main arguments:

**`req` (Request)** — the incoming data:

| Property | Source | Example |
| --- | --- | --- |
| `req.params` | Named URL segments | `/users/:id` → `req.params.id` |
| `req.query` | Query string | `/search?q=node` → `req.query.q` |
| `req.body` | Request body | Requires `express.json()` to be registered first |
| `req.headers` | HTTP headers | `req.headers.authorization` |

**`res` (Response)** — sending data back:

| Method | Use |
| --- | --- |
| `res.send()` | Text or HTML |
| `res.json()` | JSON — the default for APIs |
| `res.status(code)` | Sets the status code; chainable: `res.status(404).json({...})` |
| `res.sendFile()` | Serves a file from disk |
| `res.redirect()` | Issues a redirect |

Each request must end with **exactly one** response. Sending twice throws `ERR_HTTP_HEADERS_SENT`; sending zero times leaves the client hanging until it times out.

### Middleware Pipeline

Middleware functions intercept requests, transform them, validate credentials, or handle errors before a response is sent. They run in sequence and hand off with `next()`:

```javascript
// A simple logging middleware
app.use((req, res, next) => {
    console.log(`[${new Date().toISOString()}] ${req.method} ${req.url}`);
    next(); // Hands control to the next middleware or route
});
```

**Order matters.** Middleware registered with `app.use()` only applies to routes defined *after* it. This is the single most common source of "why isn't my `req.body` defined" — `express.json()` was registered below the route that needed it.

Forgetting `next()` (and not sending a response) silently hangs the request.

### Dynamic Route Configuration

Extract variables from the URL path using `:` placeholders:

```javascript
app.get('/api/users/:userId', (req, res) => {
    const { userId } = req.params;
    res.json({ status: 'Success', userId });
});
```

### Error Handling

Express recognizes error middleware by its **four** parameters. It must be registered last, after all routes:

```javascript
// 404 — nothing above matched
app.use((req, res) => {
    res.status(404).json({ error: 'Not found' });
});

// Error handler — note the four arguments
app.use((err, req, res, next) => {
    console.error(err.stack);
    res.status(err.status || 500).json({ error: 'Internal server error' });
});
```

In Express 4, errors thrown inside an `async` handler are **not** caught automatically — you must call `next(err)` yourself or wrap handlers. Express 5 fixes this and forwards rejected promises to the error handler for you.

### Organizing Routes with `Router`

Once you have more than a handful of endpoints, split them into modules:

```javascript
// routes/users.js
const express = require('express');
const router = express.Router();

router.get('/', getAllUsers);
router.get('/:id', getUserById);
router.post('/', createUser);

module.exports = router;

// app.js
app.use('/api/users', require('./routes/users'));
```

### Serving Static Files

```javascript
app.use(express.static('public'));
// public/style.css is now served at /style.css
```

## 📦 Middleware You'll Install Almost Every Time

| Package | Purpose |
| --- | --- |
| `cors` | Allows browser requests from other origins |
| `helmet` | Sets sensible security headers |
| `morgan` | HTTP request logging |
| `dotenv` | Loads `.env` into `process.env` |
| `express-rate-limit` | Basic abuse protection |
| `multer` | Handles file uploads / multipart forms |

Also note `express.urlencoded({ extended: true })` for parsing HTML form submissions — separate from `express.json()`.

## ⚖️ Architectural Advantages and Limits

| Pros | Cons |
| --- | --- |
| **High performance** — a thin layer that preserves Node's event-loop speed | **Lack of structure** — unopinionated design means no enforced architecture, which gets messy at scale |
| **Massive ecosystem** — enormous community and thousands of compatible packages | **Boilerplate fatigue** — you wire up validation, auth, and error handling yourself |
| **Easy to learn** — small API surface, endless tutorials | **Slower-moving** — Express 5 took nearly a decade; newer frameworks ship features faster |
| **Stable and battle-tested** — runs in production almost everywhere | **No built-in TypeScript story** — types come from `@types/express` |

## Alternatives Worth Knowing

- **Fastify** — Similar mental model, faster, with schema-based validation built in.
- **NestJS** — Opinionated and structured (modules, decorators, dependency injection); runs on Express underneath. Suits larger teams.
- **Hono** — Very lightweight, web-standard APIs, runs on edge runtimes as well as Node.

Express remains the safe default: it's what most tutorials, most job listings, and most existing codebases use.